In [6]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from tqdm import tqdm
from alerce.core import Alerce
from HGP import *
from load import *

In [7]:
csv_dir = "csv_files"

print("=" * 50)
curves_data = load_curves(csv_dir)
print(f"Loaded {len(curves_data)} curves")

print("=" * 50)

# takes VERY long time. needs to run most likely for each and every curve
fitted_gps = fit_all_gps(curves_data)
print(f"Fitted {len(fitted_gps)} GPs")

print("=" * 50)

print("Building population template...")
t_grid, template_mu, template_std, mean_noise, all_means, all_noise = build_template(fitted_gps)

print("=" * 50)

print("Scoring for precursor activity...")
results = []
for oid, gp, t, y in fitted_gps:
    scores = score_precursor(gp, t, y, t_grid, template_mu, template_std)
    scores['oid'] = oid
    results.append(scores)

results_df = pd.DataFrame(results).sort_values('mean_deviation_score', ascending=False)
precursors = results_df[results_df['is_precursor']]
print(f"\nPrecursor candidates: {len(precursors)} / {len(results_df)}")
print(precursors[['oid', 'mean_deviation_score', 'noise_excess_ratio', 'pre_peak_ll']].head(10))

NameError: name 'tqdm' is not defined

In [8]:
def predict_new(oid, t_grid, template_mu, template_std, sigma_thresh=2.5):
    alerce = Alerce()
    new_df = alerce.query_detections(oid, format="pandas")
    new_r = new_df[new_df['fid'] == 2].sort_values('mjd').drop_duplicates('mjd')
    new_g = new_df[new_df['fid'] == 1]
    
    color_offset = new_g['magpsf'].median() - new_r['magpsf'].median()
    mag = new_r['magpsf'].values + color_offset
    t = new_r['mjd'].values
    
    t_norm = (t - t.min()) / (t.max() - t.min() + 1e-9)
    mag_norm = (mag - mag.min()) / (mag.max() - mag.min() + 1e-9)
    
    gp = HeteroscedasticGP(n_iter=3).fit(t_norm, mag_norm)
    scores = score_precursor(gp, t_norm, mag_norm, t_grid, template_mu, template_std,
                             sigma_thresh=sigma_thresh)
    
    return scores['is_precursor'], scores

In [9]:
oid = "ZTF22aafmixt"
predict_new(oid, t_grid, template_mu, template_std)

(np.True_,
 {'peak_t': np.float64(0.06030150753768844),
  'mean_deviation_score': np.float64(7.2764385147023365),
  'noise_excess_ratio': np.float64(9.222153253132706e-62),
  'pre_peak_ll': np.float64(-0.2131037807264434),
  'is_precursor': np.True_})

In [10]:
file = "snii_with_precursors.csv"
df = pd.read_csv(file)
df = df[df['bands'] == 'R,g']
oids = df['ztf_oid']
for oid in oids:
    try:
        result, _ = predict_new(oid, t_grid, template_mu, template_std)
        print(f"{oid}: {result}")
    except Exception as e:
        print("didn't work")

ZTF19aaimlnn: False
ZTF20abolrpg: False
ZTF18acealtv: False
ZTF25abxlcab: True
ZTF20acxzmro: False
ZTF25acfwips: True
ZTF20aavuiwf: False
ZTF21abifctk: False
ZTF20acxztft: True
ZTF20aadyeye: False
ZTF19actlcbw: False
ZTF20acwhxah: True
ZTF19aanwfqd: False
ZTF20aafnsco: False
ZTF18acdwwql: False
ZTF19adnuazm: False


/tmp/ipykernel_1455578/3077978115.py:74: RuntimeWarning: overflow encountered in exp
  noise_var_test = np.exp(noise_mu) + 1e-6


ZTF19aargopz: False
ZTF19acvwoty: False
ZTF19abkgtka: False
ZTF18adrvopa: False
ZTF22abwtwmr: True
ZTF23absmvjg: True
ZTF19acnpwqg: True
ZTF21aadrtze: True
ZTF21abdrwiv: True
ZTF19aaiohcc: False
ZTF21acjmutc: False
ZTF23absdnkt: False
ZTF19aapfqce: True
ZTF18aazyhkf: False
ZTF20abapeub: False
ZTF19aaoiawb: True
ZTF23aablfwp: False
ZTF22abnfbik: False
